#### **FEATURE ENGINEERING**

In [63]:
import pandas as pd 
import numpy as np

catalogo_productos = pd.read_csv("Datos-finales/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-finales/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-finales/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-finales/procurement_cajas.csv")

Necesitamos averiguar el volumen total de cada producto, que nos servirá para luego calcular el porcentaje de la utilización de la caja nueva (y actual).

In [64]:
# Hacemos un merge de la tabla de producto con la de caja
df_merge = catalogo_productos.merge(
    especificaciones_cajas,
    on='caja_tipo_id',  # hacemos un join por la clave de caja
    how='left'          # dos productos distintos pueden usar la misma caja
)

# Calculamos el volumen del producto como la dimensión interna de la caja actual, pues la 
# restricción indica que su valor es el mínimo posible para contener el producto.
df_merge['volumen_producto'] = (
    df_merge['caja_interior_largo'] * 
    df_merge['caja_interior_ancho'] * 
    (df_merge['caja_interior_alto'] - 40) # Un headspace de 40mm obligatorio para la altura
)

catalogo_productos['volumen_producto'] = df_merge['volumen_producto']

Además, agregamos dos columnas extra que indican la dimensión interna de cada tipo de caja.

In [65]:
especificaciones_cajas['dimension_interna'] = (
    especificaciones_cajas['caja_interior_ancho'] *
    especificaciones_cajas['caja_interior_largo'] *
    especificaciones_cajas['caja_interior_alto']
) 

Calculamos la carga máxima soportada por cada tipo de caja (kg), cuya fórmula está dada por:

$$
carga_{\text{máx}} (\text{kg}) = \frac{ECT (\text{N/m}) \times \text{perímetro}_{\text{caja}} (\text{m})}{9.81 (m/s^2)}
$$

In [66]:
# Calculamos primero el perímetro
perimetro_cajas = (especificaciones_cajas["caja_exterior_largo"] + 
                   especificaciones_cajas["caja_exterior_ancho"]) * 2 / 1000 # Dividimos por 1000 para pasar de mm a m

# Correspondencia grosor-ect
ect_por_grosor = {2.5: 600, 2.7: 730, 3.0: 1000, 4.1: 1200, 4.5: 1400, 4.6: 1450, 4.7: 1500, 4.8: 1550, 5.0: 1650}
ects = [ect_por_grosor[g] for g in especificaciones_cajas['caja_grosor_mm']]

# Aplicamos la fórmula de carga máxima
especificaciones_cajas['carga_max'] = ects * perimetro_cajas / 9.81

Agregamos una columna que indica la carga soportada por cada producto actualmente.

In [67]:
prod_caja_merge = catalogo_productos.merge(especificaciones_cajas,
                                           on='caja_tipo_id',
                                           how='left')

# Restamos -1 para NO contar la caja de la base
catalogo_productos['carga_soportada'] = prod_caja_merge['peso_neto_caja'] * (prod_caja_merge['cantidad_cajas_alto'] - 1)

Eliminamos las columnas que no aporten valor al contexto del problema.

In [68]:
catalogo_productos.drop([
    'descripcion_producto', 'ingrediente_forma', 'tipo_proyecto', 'subcategoria',
    'categoria', 'tamaño_corte', 'tamaño_paquete'
], axis=1, inplace=True)

operaciones_planta.drop([
    'volumen_producto_canal_servicios_comida', 'volumen_producto_canal_minorista', 
    'volumen_producto_canal_cadenas_corporativas', 'volumen_producto_canal_otros'
], axis=1, inplace=True)

procurement_cajas.drop(['costo_total_base'], axis=1, inplace=True)

Exportamos las tablas modificadas en la carpeta "Datos-finales"

In [69]:
catalogo_productos.to_csv("Datos-finales/catalogo_productos.csv", index=False)
especificaciones_cajas.to_csv("Datos-finales/especificaciones_cajas.csv", index=False)
operaciones_planta.to_csv("Datos-finales/operaciones_planta.csv", index=False)
procurement_cajas.to_csv("Datos-finales/procurement_cajas.csv", index=False)